# Ad Creative Quality Scorer — PyTorch
ResNet-50 + BiLSTM + ONNX export


In [ ]:
!git clone https://github.com/saitejasrivilli/ad-creative-scorer.git
import os, sys
os.chdir("/content/ad-creative-scorer")
sys.path.insert(0, os.getcwd())
os.makedirs("models", exist_ok=True)
print(os.listdir("."))

In [ ]:
!pip install -q torch torchvision onnx onnxruntime numpy pandas scikit-learn
import torch
print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
exec(open("data/synthetic_data.py").read())
import numpy as np
df, img_feats, txt_tokens = generate_dataset(n_samples=20000)
(tr_df,tr_img,tr_txt),(va_df,va_img,va_txt),(te_df,te_img,te_txt) = train_val_test_split(df,img_feats,txt_tokens)
print(f"Train: {len(tr_df)} | Val: {len(va_df)} | Test: {len(te_df)}")

In [ ]:
exec(open("model/multitask.py").read())
model = CreativeScorer().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(["product","lifestyle","text_heavy","video_thumbnail","brand"])

def make_loader(df, img, txt, shuffle=True, bs=256):
    q = torch.tensor(df["quality_score"].values, dtype=torch.float32)
    c = torch.tensor(le.transform(df["category"].values), dtype=torch.long)
    i = torch.tensor(img, dtype=torch.float32)
    t = torch.tensor(txt, dtype=torch.long)
    return DataLoader(TensorDataset(i,t,q,c), batch_size=bs, shuffle=shuffle)

train_dl = make_loader(tr_df, tr_img, tr_txt)
val_dl   = make_loader(va_df, va_img, va_txt, shuffle=False)
test_dl  = make_loader(te_df, te_img, te_txt, shuffle=False)
print(f"Batches: train={len(train_dl)} val={len(val_dl)} test={len(test_dl)}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5)
best_corr = 0; history = {"train_loss":[], "val_corr":[]}

for epoch in range(20):
    model.train(); total_loss = 0
    for img_b,txt_b,q_b,c_b in train_dl:
        img_b,txt_b,q_b,c_b = img_b.to(device),txt_b.to(device),q_b.to(device),c_b.to(device)
        optimizer.zero_grad()
        q_pred,c_pred = model(img_b,txt_b)
        loss,_ = model.compute_loss(q_pred,c_pred,q_b,c_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    model.eval(); preds,trues = [],[]
    with torch.no_grad():
        for img_b,txt_b,q_b,c_b in val_dl:
            q_p,_ = model(img_b.to(device),txt_b.to(device))
            preds.extend(q_p.cpu().squeeze().tolist())
            trues.extend(q_b.tolist())
    import numpy as np
    corr = float(np.corrcoef(preds,trues)[0,1])
    history["train_loss"].append(total_loss/len(train_dl))
    history["val_corr"].append(corr)
    print(f"Epoch {epoch+1:2d} | loss={total_loss/len(train_dl):.4f} | val_corr={corr:.4f}")
    if corr > best_corr:
        best_corr = corr
        torch.save(model.state_dict(), "models/creative_scorer_best.pt")
        print(f"  Saved best (corr={corr:.4f})")

In [ ]:
model.load_state_dict(torch.load("models/creative_scorer_best.pt", map_location=device))
model.eval()
preds,trues,cat_preds,cat_trues = [],[],[],[]
with torch.no_grad():
    for img_b,txt_b,q_b,c_b in test_dl:
        q_p,c_p = model(img_b.to(device),txt_b.to(device))
        preds.extend(q_p.cpu().squeeze().tolist())
        trues.extend(q_b.tolist())
        cat_preds.extend(c_p.cpu().argmax(1).tolist())
        cat_trues.extend(c_b.tolist())
corr = float(np.corrcoef(preds,trues)[0,1])
cat_acc = float(np.mean(np.array(cat_preds)==np.array(cat_trues)))
print("=== TEST RESULTS ===")
print(f"  Quality correlation (r): {corr:.4f}")
print(f"  Category accuracy:       {cat_acc:.4f}")
print(f"  Best val correlation:    {best_corr:.4f}")

In [ ]:
import time
# ONNX export
dummy_img = torch.randn(1, 2048)
dummy_txt = torch.randint(0, 500, (1, 20))
torch.onnx.export(
    model.cpu(), (dummy_img, dummy_txt),
    "models/creative_scorer.onnx",
    input_names=["image_features","text_tokens"],
    output_names=["quality_score","category_logits"],
    opset_version=14,
)
print("ONNX exported")

# Benchmark PyTorch vs ONNX
model_cpu = model.cpu()
lats = []
for _ in range(200):
    t0 = time.perf_counter()
    with torch.no_grad(): _ = model_cpu(dummy_img, dummy_txt)
    lats.append((time.perf_counter()-t0)*1000)
lats.sort()
print(f"PyTorch p50={lats[100]:.3f}ms p95={lats[190]:.3f}ms p99={lats[198]:.3f}ms")

import onnxruntime as ort
sess = ort.InferenceSession("models/creative_scorer.onnx", providers=["CPUExecutionProvider"])
onnx_lats = []
img_np = dummy_img.numpy(); txt_np = dummy_txt.numpy()
for _ in range(200):
    t0 = time.perf_counter()
    _ = sess.run(None, {"image_features": img_np, "text_tokens": txt_np})
    onnx_lats.append((time.perf_counter()-t0)*1000)
onnx_lats.sort()
print(f"ONNX    p50={onnx_lats[100]:.3f}ms p95={onnx_lats[190]:.3f}ms p99={onnx_lats[198]:.3f}ms")
speedup = lats[198]/onnx_lats[198]
reduction = (1 - onnx_lats[198]/lats[198])*100
print(f"Speedup p99: {speedup:.2f}x | Latency reduction: {reduction:.1f}%")

In [ ]:
import shutil
shutil.make_archive("creative_scorer_artifacts","zip","models/")
from google.colab import files
files.download("creative_scorer_artifacts.zip")